# Nomic-embed-text-v1.5 Baseline
## Dense retrieval with task-aware instruction prefixes

**What this notebook does:**

Implements the `nomic-ai/nomic-embed-text-v1.5` dense retrieval baseline —
a 137M parameter open-source model specifically designed for long-context
and structured text retrieval.

**Why Nomic is different from MiniLM and BGE:**

Nomic uses **task instruction prefixes** — a key design choice that explicitly
tells the model what kind of text it is encoding:

```
Documents : "search_document: Company: Acme | Country: Germany | ..."
Queries   : "search_query: software companies in Germany"
```

This separation means the model knows at encoding time whether it is looking
at a company description (document) or a search request (query). This is
particularly relevant for your rich text format — Nomic was designed for exactly
this kind of structured + natural language mixed input.

**Where it sits in the lineup:**

| Model | Dims | Params | Type |
|---|---|---|---|
| BM25 | — | 0 | Sparse |
| MiniLM | 384 | 22M | Open small |
| **Nomic-v1.5** | **768** | **137M** | **Open medium** |
| BGE-large | 1024 | 335M | Open large |
| OpenAI 3-large | 3072 | — | Commercial |

Nomic fills the gap between MiniLM and BGE, giving a clean size progression.

**FAISS index:** `IndexFlatIP` (inner product) — Nomic uses normalised embeddings
like BGE, so cosine similarity via inner product is correct.

**Folder structure:**
```
result/
└── 05_baseline_nomic/
    ├── company_embeddings.npy      # Nomic 768-d embeddings (98716 x 768)
    ├── company_faiss.index         # FAISS IndexFlatIP index
    ├── nomic_results.csv           # Top-1000 results per query (101 queries)
    └── evaluation_nomic.csv        # NDCG, Precision, Recall, F1 @ k in {10,50,100,1000}
```

### Notebook structure
1. Environment setup
2. Imports
3. Load dataset & build corpus
4. GPU check
5. Encode all companies with Nomic (with `search_document:` prefix)
6. Build FAISS index
7. Run all 101 queries (with `search_query:` prefix)
8. Evaluation — NDCG, Precision, Recall, F1
9. Final summary

## 1 · Environment Setup

Create output folder automatically if it does not exist.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
API_KEY  = os.getenv('API_KEY')
BASE_URL = os.getenv('BASE_URL')

RESULT_DIR = Path('result/05_baseline_nomic_summary')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/ — ready')

## 2 · Imports

| Package | Role |
|---|---|
| `sentence_transformers` | Loads `nomic-ai/nomic-embed-text-v1.5` |
| `faiss` | Fast vector similarity search |
| `torch` | GPU detection |
| `pandas / numpy` | Data wrangling |
| `time` | Latency measurement |

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import torch
import json, time
import numpy as np
import pandas as pd
from pathlib import Path

RESULT_DIR = Path('result/05_baseline_nomic_summary')
print('[Imports] All packages loaded successfully')

## 3 · Load Dataset & Build Corpus

Same rich text format as all other baselines — identical input for fair comparison.

**One important difference from other models:**
Nomic requires a `search_document:` prefix on every company text before encoding.
This prefix is NOT part of the text content — it is an instruction to the model
telling it this text is a retrievable document, not a query.

Without this prefix, Nomic's retrieval quality drops significantly because the
model does not know what mode to operate in.

In [ ]:
print('[Load] Loading production results...')
results_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Total rows        : {len(results_df):,}')
print(f'[Load] Columns available : {list(results_df.columns)}')

all_companies = results_df.drop_duplicates(subset='domain').reset_index(drop=True)
print(f'[Load] Unique companies  : {len(all_companies):,}')

with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries           : {len(data)}')

# ── Build rich text — identical to all other baseline notebooks ───────────────
print('[Load] Building rich text for each company...')

def build_rich_text(row):
    """Summary only — Set A baseline experiments."""
    return str(row.get('summary', '')) if pd.notna(row.get('summary')) else ''

rich_texts = [build_rich_text(row) for _, row in all_companies.iterrows()]

doc_texts = [f'search_document: {t}' for t in rich_texts]

print(f'[Load] Sample document text (first company):')
print(f'  {doc_texts[0][:300]}...')
print(f'[Load] Note: search_document: prefix added to all {len(doc_texts):,} companies')

## 4 · GPU Check

Nomic-v1.5 is 137M parameters — smaller than BGE (335M) but larger than MiniLM (22M).
GPU is recommended for encoding 99k companies in reasonable time.
- CPU: ~5-8 minutes
- A100 GPU: ~2-3 minutes

In [ ]:
print(f'[GPU] CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[GPU] Device          : {torch.cuda.get_device_name(0)}')
    print(f'[GPU] VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    DEVICE = 'cuda'
else:
    print('[GPU] No GPU — running on CPU (~5-8 minutes for encoding)')
    DEVICE = 'cpu'
print(f'[GPU] Using device    : {DEVICE}')

## 5 · Encode All Companies with Nomic

**Key encoding details:**
- Prefix: `search_document:` added to every company text (already done in §3)
- `normalize_embeddings=True` — Nomic uses cosine similarity like BGE
- Output: 768-dimensional unit vectors
- Index type: `IndexFlatIP` (same as BGE — inner product = cosine for unit vectors)

**Embeddings are saved immediately** — skip re-encoding on restart.

In [5]:
embeddings_path = RESULT_DIR / 'company_embeddings.npy'

if embeddings_path.exists():
    # ── Load existing embeddings — skip API/encoding cost ────────────────────
    print('[Encode] Embeddings already exist — loading from disk...')
    embeddings  = np.load(embeddings_path).astype('float32')
    ENCODE_TIME = 0.0
    print(f'[Encode] Loaded shape  : {embeddings.shape}')
    print(f'[Encode] Dims          : {embeddings.shape[1]}')

else:
    # ── Encode from scratch ───────────────────────────────────────────────────
    print('[Encode] Loading Nomic model...')
    print('[Encode] First run downloads ~550MB from HuggingFace...')
    t0    = time.time()
    model = SentenceTransformer(
        'nomic-ai/nomic-embed-text-v1.5',
        device=DEVICE,
        trust_remote_code=True  # required for Nomic
    )
    print(f'[Encode] Model loaded in  : {time.time()-t0:.1f}s  on {model.device}')
    print(f'[Encode] Embedding dims   : 768')
    print(f'[Encode] Parameters       : ~137M')

    print('[Encode] Encoding all companies...')
    print('[Encode] Using search_document: prefix on all texts')
    batch_size = 256 if DEVICE == 'cuda' else 64
    print(f'[Encode] Batch size       : {batch_size}')
    t0 = time.time()

    embeddings = model.encode(
        doc_texts,                  # texts already have search_document: prefix
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # REQUIRED — Nomic uses cosine similarity
    )

    ENCODE_TIME = time.time() - t0
    print(f'[Encode] Done in          : {ENCODE_TIME/60:.1f} minutes')
    print(f'[Encode] Embeddings shape : {embeddings.shape}')  # (98716, 768)

    # Verify normalisation
    norms = np.linalg.norm(embeddings[:5], axis=1)
    print(f'[Encode] Sample norms     : {norms.tolist()}  (should all be ~1.0)')

    np.save(embeddings_path, embeddings)
    print(f'[Encode] Saved to         : result/05_baseline_nomic/company_embeddings.npy')

KeyboardInterrupt: 

## 6 · Build FAISS Index

**Index type: `IndexFlatIP`** — same as BGE and OpenAI.
Nomic uses cosine similarity on normalised vectors, so inner product is correct.

> **Quick reminder of index types across all baselines:**
> - MiniLM → `IndexFlatL2` (not normalised, Euclidean distance)
> - BGE, Nomic, OpenAI → `IndexFlatIP` (normalised, cosine = inner product)

In [ ]:
print('[FAISS] Loading embeddings...')
embeddings = np.load(RESULT_DIR / 'company_embeddings.npy').astype('float32')
print(f'[FAISS] Embeddings shape  : {embeddings.shape}')

print('[FAISS] Building IndexFlatIP...')
t0        = time.time()
dimension = embeddings.shape[1]  # 768
index     = faiss.IndexFlatIP(dimension)
index.add(embeddings)
INDEX_BUILD_TIME = time.time() - t0

print(f'[FAISS] Index built in    : {INDEX_BUILD_TIME:.2f}s')
print(f'[FAISS] Vectors in index  : {index.ntotal:,}')
print(f'[FAISS] Index type        : IndexFlatIP (cosine similarity)')
print(f'[FAISS] Vector dims       : {dimension}')

faiss.write_index(index, str(RESULT_DIR / 'company_faiss.index'))
print(f'[FAISS] Index saved to    : result/05_baseline_nomic/company_faiss.index')

## 7 · Run Nomic Across All 101 Queries

**Critical: queries must use `search_query:` prefix**

This is the other half of Nomic's task instruction design.
At query time, every query string gets a `search_query:` prefix — this is
different from the `search_document:` prefix used for company texts.

```
Company text : "search_document: Company: Acme | Country: Germany | ..."
Query        : "search_query: software companies in Germany"
```

Using the wrong prefix (or no prefix) will significantly hurt retrieval quality
because the model is optimised for this asymmetric document-query setup.

**Output:** `result/05_baseline_nomic/nomic_results.csv`

In [ ]:
# Load model if not already loaded (e.g. if embeddings were loaded from disk)
if 'model' not in dir() or model is None:
    print('[Run] Loading Nomic model for query encoding...')
    model = SentenceTransformer(
        'nomic-ai/nomic-embed-text-v1.5',
        device=DEVICE,
        trust_remote_code=True
    )
    print(f'[Run] Model loaded on {model.device}')

print(f'[Run] Starting Nomic retrieval for {len(data)} queries...')
print(f'[Run] Retrieving top-1000 per query')
print(f'[Run] Using search_query: prefix on all queries')
print('-' * 60)

all_results = []
query_times = []
total_start = time.time()

for i, item in enumerate(data):
    query_id = item['query_id']
    query    = item['query']

    # ── Add search_query: prefix — REQUIRED for Nomic queries ────────────────
    query_with_prefix = f'search_query: {query}'

    # ── Encode query + search — time both together ────────────────────────────
    t0       = time.perf_counter()
    q_emb    = model.encode(
        [query_with_prefix],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')
    scores, idxs = index.search(q_emb, 1000)
    query_ms = (time.perf_counter() - t0) * 1000
    query_times.append(query_ms)

    for rank, (idx, score) in enumerate(zip(idxs[0], scores[0])):
        company = all_companies.iloc[idx]
        all_results.append({
            'query_id': query_id,
            'query':    query,
            'rank':     rank + 1,
            'score':    float(score),
            'domain':   company['domain'],
            'name':     company.get('name', ''),
            'country':  company.get('country', ''),
            'summary':  company.get('summary', ''),
        })

    if (i + 1) % 20 == 0 or (i + 1) == len(data):
        elapsed   = time.time() - total_start
        remaining = (len(data) - i - 1) * elapsed / (i + 1)
        print(f'[Run] {i+1:3d}/{len(data)}  |  '
              f'avg {sum(query_times)/len(query_times):.1f}ms/query  |  '
              f'~{remaining:.0f}s remaining')

nomic_df = pd.DataFrame(all_results)
nomic_df.to_csv(RESULT_DIR / 'nomic_results.csv', index=False)

AVG_LATENCY_MS = sum(query_times) / len(query_times)
print('-' * 60)
print(f'[Run] Done!')
print(f'[Run] Total results       : {len(nomic_df):,}')
print(f'[Run] Avg query latency   : {AVG_LATENCY_MS:.1f}ms')
print(f'[Run] Saved to            : result/05_baseline_nomic/nomic_results.csv')

## 8 · Evaluation — NDCG, Precision, Recall, F1 @ k

Same evaluation protocol as all other baseline notebooks.

### Pseudo-relevance labels
A company is **relevant** for a query if it appears in the **production top-100**.

### Metrics at k ∈ {10, 50, 100, 1000}

| Metric | What it measures |
|---|---|
| **NDCG@k** | Ranking quality — rewards relevant results ranked higher |
| **Precision@k** | Of top-k results, what fraction are relevant? |
| **Recall@k** | Of all relevant companies, what fraction did we find? |
| **F1@k** | Harmonic mean of Precision and Recall |

In [ ]:
print('[Eval] Loading production labels...')
production_df = pd.read_excel('dataset/production_results.xlsx')

K_VALUES = [10, 50, 100, 500, 1000]

def get_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(
        1 / np.log2(i + 2)
        for i, d in enumerate(retrieved[:k]) if d in relevant
    )

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

print('[Eval] Computing metrics for all queries...')
eval_rows = []

for i, item in enumerate(data):
    qid       = item['query_id']
    query     = item['query']
    relevant  = get_relevant(qid)
    retrieved = (
        nomic_df[nomic_df['query_id'] == qid]
        .sort_values('rank')['domain'].tolist()
    )
    for k in K_VALUES:
        eval_rows.append({
            'query_id':  qid,
            'query':     query,
            'k':         k,
            'precision': precision_at_k(retrieved, relevant, k),
            'recall':    recall_at_k(retrieved, relevant, k),
            'f1':        f1_at_k(retrieved, relevant, k),
            'ndcg':      ndcg_at_k(retrieved, relevant, k),
        })

    if (i + 1) % 25 == 0:
        print(f'[Eval] {i+1}/101 queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_nomic.csv', index=False)
print(f'[Eval] Saved to result/05_baseline_nomic/evaluation_nomic.csv')

## 9 · Final Summary

Full results averaged across all 101 queries.

**What to watch for:**
Nomic's task prefix design should help it handle the structured rich text format
better than BGE. If Nomic outperforms BGE, it suggests the task-aware encoding
is more important than raw model size for this type of semi-structured input.
If MiniLM still wins, the finding is even stronger — small models with simple
encoding outperform larger ones on structured metadata.

In [ ]:
print('[Summary] ============================================================')
print('[Summary] Nomic-embed-text-v1.5 RESULTS')
print('[Summary] ============================================================')
print(f'\n[Summary] Encoding time     : {ENCODE_TIME/60:.1f} min' if ENCODE_TIME > 0
      else '[Summary] Encoding time     : loaded from disk')
print(f'[Summary] Index build time  : {INDEX_BUILD_TIME:.2f}s')
print(f'[Summary] Avg query latency : {AVG_LATENCY_MS:.1f}ms')
print(f'[Summary] Companies encoded : {len(all_companies):,}')
print(f'[Summary] Embedding dims    : 768')
print(f'[Summary] FAISS index type  : IndexFlatIP')
print(f'[Summary] Normalisation     : True (normalize_embeddings=True)')
print(f'[Summary] Task prefix docs  : search_document:')
print(f'[Summary] Task prefix query : search_query:')
print(f'[Summary] Queries evaluated : {len(data)}')
print()
print(f'  {"k":<6} {"NDCG":>8} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-' * 46)
for k in K_VALUES:
    sub = eval_df[eval_df['k'] == k]
    print(f'  {k:<6} '
          f'{sub["ndcg"].mean():>8.3f} '
          f'{sub["precision"].mean():>10.3f} '
          f'{sub["recall"].mean():>8.3f} '
          f'{sub["f1"].mean():>8.3f}')